# 00 Prepare Data

            Load real HuggingFace datasets and write normalized JSONL/CSV files under `slm_limits_data/`.

In [ ]:
from pathlib import Path
import csv
import json
import random

DATA_DIR = Path("slm_limits_data")
MAX_SAMPLES_PER_DATASET = 1000
SEED = 42
LOAD_FROM_HF = True
USE_FALLBACK_IF_LOAD_FAILS = False
PRESERVE_EXISTING_DATA = False
REQUIRE_REAL_DATA = True
ALLOW_FALLBACK_DATA = False

GSM8K_SPLIT = "test"
HOTPOTQA_SPLIT = "validation"
TRIVIAQA_SPLIT = "validation"

DATA_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

In [ ]:
def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_csv(path, rows):
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    fieldnames = sorted({key for row in rows for key in row})
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value for key, value in row.items()})


def count_jsonl(path):
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def looks_like_fallback(path):
    if not path.exists():
        return False
    with path.open("r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return bool(rows) and all(row.get("metadata", {}).get("source") == "local_fallback" for row in rows)


def save_dataset(name, rows):
    jsonl_path = DATA_DIR / f"{name}.jsonl"
    csv_path = DATA_DIR / f"{name}.csv"
    if PRESERVE_EXISTING_DATA and jsonl_path.exists() and csv_path.exists():
        if REQUIRE_REAL_DATA and looks_like_fallback(jsonl_path):
            raise ValueError(f"{jsonl_path} is fallback/toy data. Restore real Kaggle/HF data before real smoke.")
        return {"dataset": name, "n": count_jsonl(jsonl_path), "status": "kept_existing"}
    fallback_rows = bool(rows) and all(row.get("metadata", {}).get("source") == "local_fallback" for row in rows)
    if fallback_rows and not ALLOW_FALLBACK_DATA:
        raise ValueError(
            f"{jsonl_path} would be created from fallback/toy data. Keep ALLOW_FALLBACK_DATA = False for real smoke."
        )
    limited = rows[:MAX_SAMPLES_PER_DATASET]
    write_jsonl(jsonl_path, limited)
    write_csv(csv_path, limited)
    return {"dataset": name, "n": len(limited), "status": "written"}

In [ ]:
def fallback_reasoning():
    problems = [
        ("gsm8k_local_000", "Lena has 12 apples and gives 5 away. How many apples are left?", "7"),
        ("gsm8k_local_001", "A box has 4 rows of 6 pencils. How many pencils are there?", "24"),
        ("gsm8k_local_002", "Tom buys 3 notebooks for 2 dollars each. What is the total cost?", "6"),
        ("gsm8k_local_003", "There are 18 birds. 7 fly away. How many remain?", "11"),
        ("gsm8k_local_004", "Mia reads 8 pages per day for 5 days. How many pages does she read?", "40"),
        ("gsm8k_local_005", "A train has 9 cars with 10 seats each. How many seats are there?", "90"),
        ("gsm8k_local_006", "Sam had 30 stickers and split them equally among 5 friends. How many per friend?", "6"),
        ("gsm8k_local_007", "A shop sold 14 red pens and 9 blue pens. How many pens were sold?", "23"),
        ("gsm8k_local_008", "Nora saves 4 dollars each week for 6 weeks. How much does she save?", "24"),
        ("gsm8k_local_009", "A recipe uses 2 eggs per cake. How many eggs for 7 cakes?", "14"),
        ("gsm8k_local_010", "Jay runs 3 miles each morning for 4 mornings. How many miles?", "12"),
        ("gsm8k_local_011", "A class has 21 students. 3 groups are equal size. How many students per group?", "7"),
    ]
    return [
        {
            "sample_id": sample_id,
            "axis": "reasoning",
            "dataset": "gsm8k",
            "question": question,
            "context": "",
            "gold_answer": answer,
            "supporting_facts": [],
            "metadata": {"source": "local_fallback"},
        }
        for sample_id, question, answer in problems
    ]


def fallback_knowledge():
    facts = [
        ("knowledge_local_000", "What is the capital of France?", "Paris", "France's capital city is Paris."),
        ("knowledge_local_001", "Who wrote Pride and Prejudice?", "Jane Austen", "Pride and Prejudice is a novel by Jane Austen."),
        ("knowledge_local_002", "What planet is known as the Red Planet?", "Mars", "Mars is often called the Red Planet."),
        ("knowledge_local_003", "What gas do plants absorb from the atmosphere?", "carbon dioxide", "Plants absorb carbon dioxide during photosynthesis."),
        ("knowledge_local_004", "Who painted the Mona Lisa?", "Leonardo da Vinci", "The Mona Lisa was painted by Leonardo da Vinci."),
        ("knowledge_local_005", "What is the largest ocean on Earth?", "Pacific Ocean", "The Pacific Ocean is the largest ocean on Earth."),
        ("knowledge_local_006", "What currency is used in Japan?", "yen", "Japan uses the yen as its currency."),
        ("knowledge_local_007", "What is H2O commonly called?", "water", "H2O is the chemical formula for water."),
        ("knowledge_local_008", "Who developed the theory of relativity?", "Albert Einstein", "Albert Einstein developed the theory of relativity."),
        ("knowledge_local_009", "What is the tallest mountain above sea level?", "Mount Everest", "Mount Everest is the tallest mountain above sea level."),
        ("knowledge_local_010", "What language is primarily spoken in Brazil?", "Portuguese", "Portuguese is the primary language spoken in Brazil."),
        ("knowledge_local_011", "What is the boiling point of water at sea level in Celsius?", "100", "Water boils at 100 degrees Celsius at sea level."),
    ]
    return [
        {
            "sample_id": sample_id,
            "axis": "knowledge",
            "dataset": "triviaqa",
            "question": question,
            "context": context,
            "gold_answer": answer,
            "supporting_facts": [context],
            "metadata": {"source": "local_fallback"},
        }
        for sample_id, question, answer, context in facts
    ]


def fallback_context():
    rows = [
        ("context_local_000", "Which city hosted the 2012 Summer Olympics?", "London", "The 2012 Summer Olympics were hosted in London. Paris hosted the 2024 Summer Olympics."),
        ("context_local_001", "Which scientist discovered penicillin?", "Alexander Fleming", "Alexander Fleming discovered penicillin in 1928. Marie Curie studied radioactivity."),
        ("context_local_002", "What company created the iPhone?", "Apple", "Apple introduced the first iPhone in 2007. Samsung also makes smartphones."),
        ("context_local_003", "Which country is Kyoto located in?", "Japan", "Kyoto is a city in Japan. Seoul is a city in South Korea."),
        ("context_local_004", "What instrument has keys, pedals, and strings?", "piano", "A piano has keys, pedals, and strings. A violin has strings but no keys."),
        ("context_local_005", "Which element has the chemical symbol O?", "oxygen", "O is the chemical symbol for oxygen. Au is the symbol for gold."),
        ("context_local_006", "What animal is the national symbol of New Zealand rugby?", "kiwi", "The kiwi is a national symbol of New Zealand. The silver fern is also associated with New Zealand sport."),
        ("context_local_007", "Which city is the Eiffel Tower in?", "Paris", "The Eiffel Tower is located in Paris. The Colosseum is in Rome."),
        ("context_local_008", "Who is the Greek god of the sea?", "Poseidon", "Poseidon is the Greek god of the sea. Zeus is associated with the sky."),
        ("context_local_009", "What metal is liquid at room temperature?", "mercury", "Mercury is a metal that is liquid at room temperature. Iron is solid at room temperature."),
        ("context_local_010", "Which organ pumps blood through the body?", "heart", "The heart pumps blood through the body. The lungs exchange oxygen and carbon dioxide."),
        ("context_local_011", "What is the main ingredient in guacamole?", "avocado", "Guacamole is mainly made from avocado. Salsa is commonly made from tomato."),
    ]
    return [
        {
            "sample_id": sample_id,
            "axis": "context",
            "dataset": "hotpotqa",
            "question": question,
            "context": context,
            "gold_answer": answer,
            "supporting_facts": [context.split(".")[0] + "."],
            "metadata": {"source": "local_fallback"},
        }
        for sample_id, question, answer, context in rows
    ]

In [ ]:
def load_dataset_split(path, name=None, split="train"):
    if not LOAD_FROM_HF:
        raise RuntimeError("LOAD_FROM_HF is False.")
    try:
        from datasets import load_dataset
    except ImportError as exc:
        raise ImportError("Install the HuggingFace datasets package first: pip install datasets") from exc

    kwargs = {"split": split}
    if name is None:
        dataset = load_dataset(path, **kwargs)
    else:
        dataset = load_dataset(path, name, **kwargs)
    return dataset.shuffle(seed=SEED).select(range(min(MAX_SAMPLES_PER_DATASET, len(dataset))))


def gsm8k_gold(answer):
    marker = "####"
    if marker in str(answer):
        return str(answer).split(marker)[-1].strip().replace(",", "")
    return str(answer).strip()


def prepare_gsm8k():
    dataset = load_dataset_split("gsm8k", "main", GSM8K_SPLIT)
    rows = []
    for index, item in enumerate(dataset):
        sample_id = item.get("id") or f"gsm8k_{GSM8K_SPLIT}_{index:06d}"
        rows.append(
            {
                "sample_id": str(sample_id),
                "axis": "reasoning",
                "dataset": "gsm8k",
                "question": str(item["question"]).strip(),
                "context": "",
                "gold_answer": gsm8k_gold(item["answer"]),
                "supporting_facts": [],
                "metadata": {"source": "huggingface", "hf_dataset": "gsm8k/main", "split": GSM8K_SPLIT},
            }
        )
    return rows


def flatten_hotpot_context(context):
    if isinstance(context, dict):
        titles = context.get("title", [])
        sentence_groups = context.get("sentences", [])
        paragraphs = []
        for title, sentences in zip(titles, sentence_groups):
            text = " ".join(str(sentence) for sentence in sentences)
            paragraphs.append(f"{title}: {text}")
        return "\n\n".join(paragraphs)
    return str(context or "")


def prepare_hotpotqa():
    dataset = load_dataset_split("hotpot_qa", "distractor", HOTPOTQA_SPLIT)
    rows = []
    for index, item in enumerate(dataset):
        sample_id = item.get("id") or f"hotpotqa_{HOTPOTQA_SPLIT}_{index:06d}"
        rows.append(
            {
                "sample_id": str(sample_id),
                "axis": "context",
                "dataset": "hotpotqa",
                "question": str(item["question"]).strip(),
                "context": flatten_hotpot_context(item.get("context", "")),
                "gold_answer": str(item["answer"]).strip(),
                "supporting_facts": item.get("supporting_facts", {}),
                "metadata": {"source": "huggingface", "hf_dataset": "hotpot_qa/distractor", "split": HOTPOTQA_SPLIT},
            }
        )
    return rows


def trivia_answer(answer):
    if isinstance(answer, dict):
        value = answer.get("value")
        if value:
            return str(value).strip()
        aliases = answer.get("aliases") or []
        if aliases:
            return str(aliases[0]).strip()
    return str(answer).strip()


def prepare_triviaqa():
    dataset = load_dataset_split("trivia_qa", "rc.nocontext", TRIVIAQA_SPLIT)
    rows = []
    for index, item in enumerate(dataset):
        sample_id = item.get("question_id") or item.get("id") or f"triviaqa_{TRIVIAQA_SPLIT}_{index:06d}"
        rows.append(
            {
                "sample_id": str(sample_id),
                "axis": "knowledge",
                "dataset": "triviaqa",
                "question": str(item["question"]).strip(),
                "context": "",
                "gold_answer": trivia_answer(item.get("answer", "")),
                "supporting_facts": [],
                "metadata": {"source": "huggingface", "hf_dataset": "trivia_qa/rc.nocontext", "split": TRIVIAQA_SPLIT},
            }
        )
    return rows

In [ ]:
if LOAD_FROM_HF:
    try:
        reasoning_rows = prepare_gsm8k()
        knowledge_rows = prepare_triviaqa()
        context_rows = prepare_hotpotqa()
    except Exception:
        if not USE_FALLBACK_IF_LOAD_FAILS:
            raise
        reasoning_rows = fallback_reasoning()
        knowledge_rows = fallback_knowledge()
        context_rows = fallback_context()
else:
    if not ALLOW_FALLBACK_DATA:
        raise ValueError("LOAD_FROM_HF is False and ALLOW_FALLBACK_DATA is False. Enable HuggingFace loading for real smoke.")
    reasoning_rows = fallback_reasoning()
    knowledge_rows = fallback_knowledge()
    context_rows = fallback_context()

robustness_gsm8k = [{**row, "axis": "robustness", "dataset": "gsm8k"} for row in reasoning_rows]
robustness_hotpotqa = [{**row, "axis": "robustness", "dataset": "hotpotqa"} for row in context_rows]

summary = [
    save_dataset("reasoning_gsm8k", reasoning_rows),
    save_dataset("knowledge_data", knowledge_rows),
    save_dataset("context_hotpotqa", context_rows),
    save_dataset("robustness_gsm8k", robustness_gsm8k),
    save_dataset("robustness_hotpotqa", robustness_hotpotqa),
]
write_csv(DATA_DIR / "prepare_data_summary.csv", summary)
summary